<a href="https://colab.research.google.com/github/munnurumahesh03-coder/LLM-Powered-Web-Automation-RPA-Agent/blob/main/LLM_Web_Automation_RPA_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1: Install RPA & AI Dependencies
print("⏳ Installing Selenium, Headless Chrome, PyMongo, and Groq...")

# 1. Install Chrome and ChromeDriver (Linux level for the cloud server)
!apt-get update -qq
!apt-get install -y -qq chromium-chromedriver

# 2. Install Python Libraries
!pip install -q selenium pymongo groq

print("✅ All Automation & AI Libraries Installed Successfully!")

⏳ Installing Selenium, Headless Chrome, PyMongo, and Groq...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ All Automation & AI Libraries Installed Successfully!


In [5]:
# Cell 2: The Bulletproof Colab Web Scraper
import os
import time

print("🔧 Installing Official Google Chrome (Bypassing Colab's broken packages)...")
os.system("wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -" )
os.system("echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' >> /etc/apt/sources.list.d/google-chrome.list" )
os.system("apt-get update -qq")
os.system("apt-get install -y -qq google-chrome-stable")

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

print("🤖 Booting up the Headless RPA Robot...")

# 1. Configure the Headless Browser
chrome_options = Options()
chrome_options.add_argument('--headless=new')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# 2. Start the Browser (Selenium 4 automatically downloads the right driver!)
driver = webdriver.Chrome(options=chrome_options)

# 3. Navigate to the Medical Directory
mock_url = """data:text/html,
<html><body>
<h1>HextGen MedTech - Public Provider Directory</h1>
<div class="provider">
    <p>Dr. Sarah Jenkins - Cardiology</p>
    <p>Contact: 555-0198</p>
    <p>Available: Mon-Wed</p>
</div>
<div class="provider">
    <p>Dr. Marcus Chen, Neurology</p>
    <p>Phone: (555) 847-3321</p>
    <p>Notes: Not accepting new patients.</p>
</div>
<div class="provider">
    <p>Pediatrics: Dr. Emily Ross</p>
    <p>Call 555-0024 for appointments.</p>
</div>
</body></html>
"""

print("🌐 Navigating to the Medical Directory...")
driver.get(mock_url)
time.sleep(2) # Wait for the page to fully render

# 4. Scrape the raw, messy text from the website's body
raw_text = driver.find_element(By.TAG_NAME, "body").text

print("\n✅ Scraping Complete! Here is the raw, unstructured data we extracted:\n")
print("=" * 50)
print(raw_text)
print("=" * 50)

# 5. Close the browser
driver.quit()

🔧 Installing Official Google Chrome (Bypassing Colab's broken packages)...
🤖 Booting up the Headless RPA Robot...
🌐 Navigating to the Medical Directory...

✅ Scraping Complete! Here is the raw, unstructured data we extracted:

HextGen MedTech - Public Provider Directory
Dr. Sarah Jenkins - Cardiology
Contact: 555-0198
Available: Mon-Wed
Dr. Marcus Chen, Neurology
Phone: (555) 847-3321
Notes: Not accepting new patients.
Pediatrics: Dr. Emily Ross
Call 555-0024 for appointments.


In [6]:
# Cell 3: The LLM Brain (Structuring the Data)
import os
import getpass
import json
from groq import Groq

print("🧠 Waking up the LLM Brain...")

# 1. Securely enter your Groq API Key
if "GROQ_API_KEY" not in os.environ:
    print("🔑 Enter your Groq API Key:")
    os.environ["GROQ_API_KEY"] = getpass.getpass()

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# 2. The Prompt Engineering Magic
prompt = f"""
You are an elite Data Extraction AI.
I will give you raw, messy text scraped from a medical directory.
Your job is to extract the doctors' information and format it as a strict JSON array.

Extract these exact fields for each doctor:
- "name" (string)
- "specialty" (string)
- "phone" (string)
- "notes" (string, or null if none)

RAW TEXT:
{raw_text}

OUTPUT ONLY VALID JSON. Do not include any markdown formatting, explanations, or backticks. Just the raw JSON array starting with [ and ending with ].
"""

print("⚙️ Processing unstructured data into JSON...")

# 3. Call the Groq API (Using the 120B model for maximum accuracy)
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)

# 4. Parse and display the result
structured_data = response.choices[0].message.content.strip()

# Clean up just in case the LLM adds markdown backticks
if structured_data.startswith("```json"):
    structured_data = structured_data[7:-3].strip()
elif structured_data.startswith("```"):
    structured_data = structured_data[3:-3].strip()

# Convert the text string into an actual Python Dictionary/List
json_data = json.loads(structured_data)

print("\n🎯 AI Extraction Complete! Here is the clean JSON data ready for MongoDB:\n")
print(json.dumps(json_data, indent=4))

🧠 Waking up the LLM Brain...
🔑 Enter your Groq API Key:
··········
⚙️ Processing unstructured data into JSON...

🎯 AI Extraction Complete! Here is the clean JSON data ready for MongoDB:

[
    {
        "name": "Sarah Jenkins",
        "specialty": "Cardiology",
        "phone": "555-0198",
        "notes": null
    },
    {
        "name": "Marcus Chen",
        "specialty": "Neurology",
        "phone": "(555) 847-3321",
        "notes": "Not accepting new patients."
    },
    {
        "name": "Emily Ross",
        "specialty": "Pediatrics",
        "phone": "555-0024",
        "notes": null
    }
]


In [9]:
# Cell 4: Push Data to MongoDB (AWS Cloud)
import os
import getpass
from pymongo import MongoClient

# Clear the old bad link from memory so it asks you again!
if "MONGO_URI" in os.environ:
    del os.environ["MONGO_URI"]

print("☁️ Connecting to MongoDB Atlas (AWS Cluster)...")

# 1. Securely enter your FIXED MongoDB Connection String
if "MONGO_URI" not in os.environ:
    print("🔗 Enter your FIXED MongoDB Connection String:")
    os.environ["MONGO_URI"] = getpass.getpass()

try:
    # 2. Connect to the Cloud Cluster
    client = MongoClient(os.environ["MONGO_URI"])

    # 3. Create/Select the Database and Collection (Table)
    db = client["hextgen_medtech"]
    collection = db["providers"]

    # 4. Clear old data (Just for testing)
    collection.delete_many({})

    # 5. Insert the AI-cleaned JSON data!
    print("🚀 Pushing data to the cloud...")
    result = collection.insert_many(json_data)

    print(f"\n✅ SUCCESS! {len(result.inserted_ids)} records securely pushed to your AWS MongoDB Database!")

except Exception as e:
    print(f"\n❌ ERROR connecting to MongoDB: {e}")

☁️ Connecting to MongoDB Atlas (AWS Cluster)...
🔗 Enter your FIXED MongoDB Connection String:
··········
🚀 Pushing data to the cloud...

✅ SUCCESS! 3 records securely pushed to your AWS MongoDB Database!


In [10]:
# Cell 5: Generate Production Files for GitHub
import os
from google.colab import files

print("📦 Packaging the RPA Agent for Production...")

# 1. Write the Python Script
with open("rpa_agent.py", "w") as f:
    f.write("""import os
import json
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from groq import Groq
from pymongo import MongoClient

print("🤖 RPA Agent Waking Up...")

# 1. Configure Headless Browser
chrome_options = Options()
chrome_options.add_argument('--headless=new')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
driver = webdriver.Chrome(options=chrome_options)

# 2. Scrape the Website
mock_url = "data:text/html,<html><body><h1>HextGen MedTech - Public Provider Directory</h1><div class='provider'><p>Dr. Sarah Jenkins - Cardiology</p><p>Contact: 555-0198</p><p>Available: Mon-Wed</p></div><div class='provider'><p>Dr. Marcus Chen, Neurology</p><p>Phone: (555) 847-3321</p><p>Notes: Not accepting new patients.</p></div><div class='provider'><p>Pediatrics: Dr. Emily Ross</p><p>Call 555-0024 for appointments.</p></div></body></html>"
driver.get(mock_url)
time.sleep(2)
raw_text = driver.find_element(By.TAG_NAME, "body").text
driver.quit()
print("✅ Data Scraped!")

# 3. Process with LLM
client = Groq(api_key=os.environ["GROQ_API_KEY"])
prompt = f"Extract doctors info as strict JSON array with keys: name, specialty, phone, notes. RAW TEXT:\\n{raw_text}\\nOUTPUT ONLY VALID JSON."
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)
structured_data = response.choices[0].message.content.strip()
if structured_data.startswith("```json"): structured_data = structured_data[7:-3].strip()
elif structured_data.startswith("```"): structured_data = structured_data[3:-3].strip()
json_data = json.loads(structured_data)
print("✅ Data Cleaned by AI!")

# 4. Push to MongoDB
mongo_client = MongoClient(os.environ["MONGO_URI"])
db = mongo_client["hextgen_medtech"]
collection = db["providers"]
collection.delete_many({}) # Clear old data for the daily refresh
collection.insert_many(json_data)
print(f"✅ SUCCESS! {len(json_data)} records pushed to MongoDB Atlas!")
""")

# 2. Write the Requirements File
with open("requirements.txt", "w") as f:
    f.write("selenium==4.24.0\ngroq==0.11.0\npymongo==4.8.0\n")

# 3. Download to your laptop
print("📥 Downloading files...")
files.download("rpa_agent.py")
files.download("requirements.txt")

📦 Packaging the RPA Agent for Production...
📥 Downloading files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>